# Colab Big Wins for Circuit Breakdown

This notebook is for quick GPU wins, not for changing the frozen study. It runs fresh outputs in a Colab work copy and keeps the original repo/results untouched.

Best quick wins:
1. Powered capability check: larger HellaSwag/ARC samples for G4.
2. Independent CUDA replay of convergence and guard/rate pilots for G1.
3. Fast task-validity diagnostics for G2 with saved model answers.

## Before You Run

Put the project in Google Drive like this:

```text
MyDrive/SLM_PAPER/
  circuit_breakdown/
  phi-3.5-mini/
  llama-3.2-3b/
  nemotron-mini-4b/
```

The model folders must be siblings of `circuit_breakdown`, because the existing protocols refer to models by those folder names.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

RUN_TAG = '20260908_colab_big_wins'
DRIVE_SLM = Path('/content/drive/MyDrive/SLM_PAPER')
SOURCE_PROJECT = DRIVE_SLM / 'circuit_breakdown'
WORK_ROOT = Path('/content/slm_paper_colab')
PROJECT = WORK_ROOT / 'circuit_breakdown'
MODEL_NAMES = ['phi-3.5-mini', 'llama-3.2-3b', 'nemotron-mini-4b']

MANIFEST = PROJECT / 'data/validated_manifest_v1.json'
STUDY = PROJECT / 'results/validated_v1'
HELLASWAG_240 = PROJECT / 'data_bench/hellaswag_val.jsonl'
HELLASWAG_LARGE = PROJECT / 'data_bench/hellaswag_val_large.jsonl'
ARC_200 = PROJECT / 'data_bench/arc_easy_test_200.json'
ARC_LARGE = PROJECT / 'data_bench/arc_easy_test_1000.json'
TEXT = PROJECT / 'data_bench/tinyshakespeare.txt'
CAPABILITY_OUTDIR = PROJECT / f'results/{RUN_TAG}_capability_powered'

N_HELLASWAG = 1000
N_ARC = 1000
DEVICE = 'cuda'

def run(cmd, cwd=PROJECT, check=True):
    print('\n$ ' + ' '.join(map(str, cmd)))
    return subprocess.run([str(part) for part in cmd], cwd=str(cwd), check=check, text=True)

print('Configured paths:')
print('SOURCE_PROJECT =', SOURCE_PROJECT)
print('PROJECT        =', PROJECT)
print('OUTPUT         =', CAPABILITY_OUTDIR)

In [ ]:
colab = __import__('google.colab', fromlist=['drive'])
colab.drive.mount('/content/drive')

assert SOURCE_PROJECT.is_dir(), f'Missing repo folder: {SOURCE_PROJECT}'
for name in MODEL_NAMES:
    assert (DRIVE_SLM / name).is_dir(), f'Missing model folder: {DRIVE_SLM / name}'
print('Drive mounted and required top-level folders exist.')

In [ ]:
# Make a fresh Colab work copy. This avoids fixed-output collisions in pilot runners.
if PROJECT.exists():
    shutil.rmtree(PROJECT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def ignore_large_outputs(directory, names):
    if Path(directory).name == 'results':
        keep = {'validated_v1', 'postrun_audit_v1'}
        return [name for name in names if name not in keep]
    return []

shutil.copytree(SOURCE_PROJECT, PROJECT, ignore=ignore_large_outputs)

# Link model folders as siblings of circuit_breakdown, matching repo protocol paths.
for name in MODEL_NAMES:
    link = WORK_ROOT / name
    if link.exists() or link.is_symlink():
        link.unlink()
    link.symlink_to(DRIVE_SLM / name, target_is_directory=True)

print('Fresh work copy ready:', PROJECT)

In [ ]:
# Install repo dependencies. Colab usually already has CUDA Torch; this keeps Torch untouched when present.
run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.44', 'numpy>=1.26', 'tqdm>=4.66', 'datasets>=2.20'])

env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT / 'src')
env['HF_HUB_OFFLINE'] = '1'
env['TRANSFORMERS_OFFLINE'] = '1'

run([sys.executable, '-m', 'py_compile',
     'src/localize.py', 'src/run_capability_study.py', 'src/run_convergence_pilot.py',
     'src/run_guard_rate_pilot.py', 'src/run_swap_format_pilot.py', 'src/run_baseline_context_pilot.py'])
print('Python files compile.')

In [ ]:
# Path check: these are the minimum files needed for the big-win runs.
required = [
    MANIFEST, STUDY / 'phi-3.5-mini/intermediate/calibration.json',
    STUDY / 'phi-3.5-mini/transfer/calibration.json',
    PROJECT / 'results/postrun_audit_v1/reproducibility/manifest.json',
    HELLASWAG_240, ARC_200, TEXT,
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(map(str, missing)))

torch = __import__('torch')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('All required project/data paths exist.')

## Optional: Build Larger Public Benchmark Caches

Run this if you want a stronger G4 capability answer. It uses public datasets only. If download is blocked, keep the existing 240 HellaSwag and 200 ARC files.

In [ ]:
# Build a larger HellaSwag validation cache.
if not HELLASWAG_LARGE.exists():
    import json
    datasets = __import__('datasets', fromlist=['load_dataset'])
    rows = datasets.load_dataset('Rowan/hellaswag', split='validation')
    HELLASWAG_LARGE.parent.mkdir(parents=True, exist_ok=True)
    with HELLASWAG_LARGE.open('w') as stream:
        for row in rows.select(range(min(N_HELLASWAG, len(rows)))):
            stream.write(json.dumps({'ctx': row['ctx'], 'endings': row['endings'], 'label': row['label']}) + '\n')
    print('Wrote', HELLASWAG_LARGE)
else:
    print('Using existing', HELLASWAG_LARGE)

# Build a larger ARC-Easy cache through the repo's validated formatter.
if not ARC_LARGE.exists():
    run([sys.executable, 'src/public_benchmarks.py', '--output', ARC_LARGE, '--count', N_ARC])
else:
    print('Using existing', ARC_LARGE)

## Big Win 1: Powered Capability Check

This is the highest-value Colab run. It attacks G4 directly: are active-prefix edits still safe when we use more benchmark questions? Start with Phi because that is the main diagnostic model. Then run all three models if time allows.

In [ ]:
# Phi first: fastest high-value powered capability run.
run([
    sys.executable, 'src/run_capability_study.py',
    '--manifest', MANIFEST,
    '--model', WORK_ROOT / 'phi-3.5-mini',
    '--study', STUDY,
    '--outdir', CAPABILITY_OUTDIR,
    '--hellaswag', HELLASWAG_LARGE if HELLASWAG_LARGE.exists() else HELLASWAG_240,
    '--arc', ARC_LARGE if ARC_LARGE.exists() else ARC_200,
    '--text', TEXT,
    '--n-hs', N_HELLASWAG if HELLASWAG_LARGE.exists() else 240,
    '--seeds', '0', '1', '2',
    '--device', DEVICE,
])

In [ ]:
# Optional: same powered capability run for all three models.
for model_name in MODEL_NAMES:
    run([
        sys.executable, 'src/run_capability_study.py',
        '--manifest', MANIFEST,
        '--model', WORK_ROOT / model_name,
        '--study', STUDY,
        '--outdir', CAPABILITY_OUTDIR,
        '--hellaswag', HELLASWAG_LARGE if HELLASWAG_LARGE.exists() else HELLASWAG_240,
        '--arc', ARC_LARGE if ARC_LARGE.exists() else ARC_200,
        '--text', TEXT,
        '--n-hs', N_HELLASWAG if HELLASWAG_LARGE.exists() else 240,
        '--seeds', '0', '1', '2',
        '--device', DEVICE,
    ])

## Big Win 2: Independent CUDA Replay for G1

These runs are good for reviewer confidence. They do not close G1 by themselves, but they give a clean Colab replay of optimizer behavior with pre-run provenance.

In [ ]:
run([sys.executable, 'src/run_convergence_pilot.py', '--project', PROJECT, '--device', DEVICE])
run([sys.executable, 'src/run_guard_rate_pilot.py', '--project', PROJECT, '--device', DEVICE])

## Big Win 3: Fast G2 Diagnostics

These are shorter than the optimizer pilots. They save generated answers and help separate answer-format failure from real task failure.

In [ ]:
run([sys.executable, 'src/run_swap_format_pilot.py', '--project', PROJECT, '--device', DEVICE])
run([sys.executable, 'src/run_baseline_context_pilot.py', '--project', PROJECT, '--device', DEVICE])

## Collect the Outputs

Copy the fresh Colab outputs back to Drive. This keeps your local repo clean until you decide what to import.

In [ ]:
DEST = DRIVE_SLM / 'colab_outputs' / RUN_TAG
DEST.mkdir(parents=True, exist_ok=True)

for relative in [
    f'results/{RUN_TAG}_capability_powered',
    'results/convergence_pilot_v1',
    'results/guard_rate_pilot_v1',
    'results/swap_format_pilot_v1',
    'results/baseline_context_pilot_v1',
]:
    source = PROJECT / relative
    if source.exists():
        target = DEST / relative
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(source, target)
        print('Copied', source, '->', target)

print('Done. Outputs are in', DEST)

## How to Read the Win

A strong quick win is not just more rows. The best outcome is one of these:

- G4 improves: larger capability intervals become tight enough to say whether preservation is supported or not.
- G1 stabilizes: Colab replay matches the Mac diagnostic direction, giving independent confidence.
- G2 clarifies: saved answers show whether the harder task is failing because of formatting, prompt context, or actual weak reasoning.

Do not overwrite the frozen study with these outputs. Treat them as new diagnostic evidence until reviewed.